<a href="https://colab.research.google.com" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de partido — [EQUIPO_LOCAL] vs [EQUIPO_VISITANTE]
> Completar el header con los datos del partido antes de ejecutar.

**Competición:** ← reemplazar  
**Fecha:** ← reemplazar  
**Resultado:** ← reemplazar  
**Fuente de datos:** FBref  
**Autor:** Andrés  

---

## Estructura del notebook

1. Setup e importaciones  
2. Carga del dataset  
3. Limpieza  
4. EDA — Análisis exploratorio

> Los datos corresponden a las estadísticas individuales de los jugadores del partido.  
> La limpieza se realiza sobre `df_raw` y el análisis sobre `df_clean`.

## 1. Setup

### 1.1 Importaciones y configuración global

**⚠️ Única celda que cambia partido a partido — completar antes de ejecutar.**

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Supresión de warnings
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

# Configuración de visualización del DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configuración de gráficos
sns.set_theme(style='whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 13

# ── CAMBIAR ESTOS VALORES PARTIDO A PARTIDO ──────────────────────────────
EQUIPO_LOCAL     = 'Equipo Local'        # ← reemplazar
EQUIPO_VISITANTE = 'Equipo Visitante'    # ← reemplazar
RESULTADO        = 'X - X'              # ← reemplazar (ej: '1 - 0')
FECHA            = 'DD/MM/AAAA'         # ← reemplazar
COMPETICION      = 'Liga / Copa'         # ← reemplazar
DATA_PATH        = '../data/raw/NOMBRE_ARCHIVO_raw.csv'  # ← reemplazar
# ─────────────────────────────────────────────────────────────────────────

# Rutas de salida — se crean automáticamente
OUTPUT_PATH = '../outputs'
CLEAN_PATH  = '../data/clean'
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CLEAN_PATH, exist_ok=True)

# Nombre base para archivos de salida
PARTIDO_ID = f"{EQUIPO_LOCAL.lower().replace(' ', '_')}_vs_{EQUIPO_VISITANTE.lower().replace(' ', '_')}"

print('✅ Configuración lista')
print(f'   Partido: {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} | {RESULTADO} | {FECHA}')

### 1.2 Diccionarios de traducción FBref

Traducción de columnas y posiciones — no modificar.

In [ ]:
COLUMNAS_ES = {
    'Player':        'Jugador',
    '#':             'Camiseta',
    'Pos':           'Posición',
    'Pos_Principal': 'Posición principal',
    'Nation':        'País',
    'Min':           'Minutos',
    'Gls':           'Goles',
    'Ast':           'Asistencias',
    'PK':            'Penales convertidos',
    'PKatt':         'Penales intentados',
    'Sh':            'Tiros totales',
    'SoT':           'Tiros al arco',
    'CrdY':          'Tarjetas amarillas',
    'CrdR':          'Tarjetas rojas',
    'Fls':           'Faltas cometidas',
    'Fld':           'Faltas recibidas',
    'Off':           'Fueras de juego',
    'Crs':           'Centros',
    'TklW':          'Entradas ganadas',
    'Int':           'Intercepciones',
    'OG':            'Goles en contra',
    'PKwon':         'Penales ganados',
    'PKcon':         'Penales concedidos',
    'edad_anios':    'Edad (años)',
    'edad_dias':     'Edad (días)',
}

POSICIONES_ES = {
    'GK': 'Arquero',        'DF': 'Defensor',
    'MF': 'Mediocampista',  'FW': 'Delantero',
    'FB': 'Lateral',        'LB': 'Lateral izquierdo',
    'RB': 'Lateral derecho','CB': 'Defensor central',
    'DM': 'Mediocampista defensivo', 'CM': 'Mediocampista central',
    'LM': 'Mediocampista izquierdo', 'RM': 'Mediocampista derecho',
    'WM': 'Mediocampista amplio',    'LW': 'Extremo izquierdo',
    'RW': 'Extremo derecho',         'AM': 'Mediocampista ofensivo',
    'FWMF': 'Delantero/Mediocampista', 'MFFW': 'Mediocampista/Delantero',
    'DFMF': 'Defensor/Mediocampista',  'MFDF': 'Mediocampista/Defensor',
}

def traducir_columnas(df):
    """Renombra columnas FBref al español. Solo traduce las que existen."""
    columnas_presentes = {k: v for k, v in COLUMNAS_ES.items() if k in df.columns}
    df = df.rename(columns=columnas_presentes)
    print(f'Columnas traducidas: {len(columnas_presentes)}')
    return df

def traducir_posiciones(df, columna='Posición'):
    """Traduce abreviaturas de posición. Maneja posiciones combinadas (ej: DF,FW)."""
    if columna not in df.columns:
        print(f"Columna '{columna}' no encontrada")
        return df
    def traducir_pos(valor):
        if pd.isna(valor):
            return valor
        partes = str(valor).split(',')
        traducidas = [POSICIONES_ES.get(p.strip(), p.strip()) for p in partes]
        return ' / '.join(traducidas)
    df[columna] = df[columna].apply(traducir_pos)
    return df

print('✅ Diccionarios y funciones de traducción listos')

### 1.3 Función de perfil de jugador

Reutilizable en cualquier análisis — no modificar.

In [ ]:
def perfil_jugador(df: pd.DataFrame, nombre: str) -> None:
    """
    Muestra el perfil completo de un jugador comparado con el promedio de su posición.
    Indicadores: ▲ por encima | ▼ por debajo | ─ igual al promedio
    """
    jugador = df[df['Jugador'].str.contains(nombre, case=False, na=False)]

    if jugador.empty:
        print(f"Jugador '{nombre}' no encontrado")
        print(f"Jugadores disponibles: {df['Jugador'].tolist()}")
        return

    jugador = jugador.iloc[0]
    posicion = jugador['Posición principal']
    promedio_pos = df[df['Posición principal'] == posicion].select_dtypes(include='number').mean()

    metricas = ['Minutos', 'Tiros totales', 'Tiros al arco', 'Goles',
                'Asistencias', 'Entradas ganadas', 'Intercepciones',
                'Faltas cometidas', 'Faltas recibidas', 'Centros']
    metricas_presentes = [m for m in metricas if m in df.columns]

    print(f"\n{'='*60}")
    print(f"PERFIL: {jugador['Jugador']} | {posicion} | {jugador['Minutos']} min")
    print(f"Partido: {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})")
    print(f"{'='*60}")
    print(f"{'Métrica':<25} {'Valor':>8} {'Prom. posición':>15}  Ind.")
    print(f"{'-'*60}")
    for m in metricas_presentes:
        if m in jugador.index and m in promedio_pos.index:
            diff = jugador[m] - promedio_pos[m]
            ind = '▲' if diff > 0 else ('▼' if diff < 0 else '─')
            print(f"{m:<25} {jugador[m]:>8.0f} {promedio_pos[m]:>15.1f}  {ind}")
    print(f"{'='*60}")
    print('▲ por encima del promedio | ▼ por debajo | ─ igual')

print('✅ Función perfil_jugador lista')

## 2. Carga del dataset

In [ ]:
df_raw = pd.read_csv(DATA_PATH, sep=',', encoding='utf-8')
print(f'✅ Dataset cargado — Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}')
print(f'   Fuente: FBref | Partido: {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} | Fecha: {FECHA}')

## 3. Limpieza

### 3.1 Diagnóstico inicial

Primer vistazo al dataset crudo para identificar problemas de estructura, tipos y calidad.

In [ ]:
print('\n--- Primeras 5 filas ---')
display(df_raw.head())

print('\n--- Últimas 5 filas ---')
display(df_raw.tail())

print(f'\nFilas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}')

print('\n--- Info general ---')
df_raw.info()

print('\n--- Nulos por columna ---')
if df_raw.isnull().sum().sum() == 0:
    print('✅ Sin valores nulos')
else:
    nulos = pd.DataFrame({
        'Nulos': df_raw.isnull().sum(),
        'Porcentaje': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
    })
    display(nulos[nulos['Nulos'] > 0])

print(f'\nFilas duplicadas: {df_raw.duplicated().sum()}')

print('\n--- Estadísticas descriptivas ---')
display(df_raw.describe())

### 3.2 Eliminar columnas técnicas de FBref

FBref incluye columnas que no aportan valor analítico:
- `PKwon` / `PKcon`: penales ganados/concedidos — casi siempre vacías en partidos
- `-9999`: hash técnico interno de FBref

**⚠️ Si el partido tiene datos en PKwon/PKcon, comentar esas líneas.**

In [ ]:
# Columnas a eliminar — ajustar si alguna tiene datos relevantes
cols_eliminar = []

for col in ['PKwon', 'PKcon', '-9999']:
    if col in df_raw.columns:
        cols_eliminar.append(col)

if cols_eliminar:
    df_raw = df_raw.drop(columns=cols_eliminar)
    print(f'✅ Columnas eliminadas: {cols_eliminar}')
else:
    print('ℹ️  No se encontraron columnas técnicas para eliminar')

print(f'   Shape actual: {df_raw.shape}')

### 3.3 Limpiar columna Nation

FBref incluye un código de bandera junto al código de país (ej: `ar ARG`).  
Nos quedamos solo con los últimos 3 caracteres (código ISO del país).

In [ ]:
if 'Nation' in df_raw.columns:
    df_raw['Nation'] = df_raw['Nation'].str[-3:]
    print('✅ Columna Nation normalizada')
    print(df_raw['Nation'].value_counts().to_string())
else:
    print('ℹ️  Columna Nation no encontrada en este dataset')

### 3.4 Separar columna Age

FBref almacena la edad en formato `años-días` (ej: `32-352`).  
Separamos en dos columnas numéricas.

In [ ]:
if 'Age' in df_raw.columns:
    df_raw[['edad_anios', 'edad_dias']] = df_raw['Age'].str.split('-', expand=True).astype(int)

    print(f"NaN en edad_anios: {df_raw['edad_anios'].isnull().sum()}")
    print(f"NaN en edad_dias:  {df_raw['edad_dias'].isnull().sum()}")
    display(df_raw[['Age', 'edad_anios', 'edad_dias']].head(5))

    df_raw = df_raw.drop(columns=['Age'])
    print('✅ Columna Age separada y eliminada')
else:
    print('ℹ️  Columna Age no encontrada — puede que ya esté procesada')

### 3.5 Detección de outliers

Método IQR para identificar valores extremos en variables numéricas.

In [ ]:
def detectar_outliers(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers = df[(df[columna] < lim_inf) | (df[columna] > lim_sup)]
    print(f'{columna}:')
    print(f'  Rango normal: [{lim_inf:.2f}, {lim_sup:.2f}]')
    print(f'  Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)\n')

columnas_numericas_raw = df_raw.select_dtypes(include=['number']).columns

for col in columnas_numericas_raw:
    detectar_outliers(df_raw, col)
    print('-' * 40)

### 3.6 Crear columna Posición principal

FBref puede asignar posiciones híbridas (ej: `FWMF`).  
Creamos una columna con los primeros 2 caracteres para simplificar el agrupamiento.

**⚠️ Revisar si hay jugadores con posición incorrecta y corregir manualmente abajo.**

In [ ]:
df_raw['Pos_Principal'] = df_raw['Pos'].str[:2]

# ── Correcciones manuales — agregar/quitar según el partido ──────────────
# df_raw.loc[df_raw['Player'] == 'NOMBRE JUGADOR', 'Pos_Principal'] = 'XX'
# ─────────────────────────────────────────────────────────────────────────

print('✅ Posición principal creada')
print(df_raw[['Player', 'Pos', 'Pos_Principal']].to_string())

### 3.7 Traducción de columnas y posiciones

In [ ]:
# Primero traducir posiciones (mientras las columnas aún están en inglés)
df_raw = traducir_posiciones(df_raw, columna='Pos')
df_raw = traducir_posiciones(df_raw, columna='Pos_Principal')

# Después traducir nombres de columnas
df_raw = traducir_columnas(df_raw)

print('✅ Traducción aplicada')
display(df_raw.head())

### 3.8 Tipos de datos finales

In [ ]:
print('--- Tipos de datos ---')
print(df_raw.dtypes)

In [ ]:
# Conversiones necesarias — ajustar según el partido
if 'Edad (días)' in df_raw.columns:
    df_raw['Edad (días)'] = df_raw['Edad (días)'].astype(int)
    print('✅ Edad (días) → int64')

### 3.9 Reset de índice

In [ ]:
print(f'Índice antes del reset: {df_raw.index.tolist()[:10]}...')
df_raw = df_raw.reset_index(drop=True)
print(f'Índice después del reset: {df_raw.index.tolist()[:10]}...')
print(f'Filas finales: {df_raw.shape[0]}')

### 3.10 Reporte de calidad post-limpieza

In [ ]:
print('=' * 55)
print('REPORTE FINAL DE CALIDAD')
print('=' * 55)
print(f'Partido:            {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE}')
print(f'Resultado:          {RESULTADO}')
print(f'Fecha:              {FECHA}')
print(f'Competición:        {COMPETICION}')
print(f'Filas finales:      {df_raw.shape[0]}')
print(f'Columnas:           {df_raw.shape[1]}')
print(f'Duplicados:         {df_raw.duplicated().sum()}')
print(f'Nulos totales:      {df_raw.isnull().sum().sum()}')
print('=' * 55)

### 3.11 Exportar dataset limpio

In [ ]:
df_clean = df_raw.copy()

nombre_salida = f'{CLEAN_PATH}/{PARTIDO_ID}_clean.csv'
df_clean.to_csv(nombre_salida, index=False, encoding='utf-8')
print(f'✅ Dataset exportado: {nombre_salida}')
print(f'   Filas: {df_clean.shape[0]} | Columnas: {df_clean.shape[1]}')

## 4. EDA — Análisis exploratorio

### 4.1 Diagnóstico estructural

In [ ]:
print('--- Info general ---')
df_clean.info()

print('\n--- Estadísticas descriptivas ---')
display(df_clean.describe())

print('\n--- Primeras filas ---')
display(df_clean.head())

print('\n--- Nulos ---')
nulos = pd.DataFrame({
    'Nulos': df_clean.isnull().sum(),
    'Porcentaje': (df_clean.isnull().sum() / len(df_clean) * 100).round(2)
})
if nulos[nulos['Nulos'] > 0].shape[0] > 0:
    display(nulos[nulos['Nulos'] > 0])
else:
    print('✅ Sin nulos')

print(f'\nFilas duplicadas: {df_clean.duplicated().sum()}')

print('\n--- Valores únicos por columna ---')
display(df_clean.nunique().sort_values().to_frame(name='Valores únicos'))

### 4.2 Correlación entre variables numéricas

**Pregunta táctica:** ¿Qué métricas del partido se relacionan entre sí?

In [ ]:
columnas_numericas = df_clean.select_dtypes(include='number').columns.tolist()

plt.figure(figsize=(12, 10))
correlacion = df_clean[columnas_numericas].corr()

sns.heatmap(
    correlacion,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
)
plt.title(f'Correlación entre variables numéricas\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Filtro de jugadores activos

In [ ]:
df_activos = df_clean[df_clean['Minutos'] > 0].copy()
print(f'Jugadores con minutos jugados: {len(df_activos)} de {len(df_clean)} en el dataset')

### 4.4 Radar chart — Perfil ofensivo/defensivo por jugador

**Pregunta táctica:** ¿Cómo se compara el perfil integral de los jugadores más destacados?

El radar chart permite ver múltiples dimensiones de un jugador de forma simultánea.  
Cada métrica se normaliza por percentil — un valor de 100 significa el mejor del partido en esa métrica.

In [ ]:
import matplotlib.patches as mpatches
import numpy as np

def radar_jugadores(df: pd.DataFrame, jugadores: list, metricas: list) -> None:
    """
    Radar chart comparativo para múltiples jugadores.
    Las métricas se normalizan por percentil (0-100) dentro del dataset.
    """
    # Normalizar métricas a percentil
    df_norm = df.copy()
    for m in metricas:
        if m in df_norm.columns:
            df_norm[m] = df_norm[m].rank(pct=True) * 100

    N = len(metricas)
    angulos = [n / float(N) * 2 * np.pi for n in range(N)]
    angulos += angulos[:1]

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    colores = ['#1a73e8', '#e8341a', '#34a853', '#fbbc04', '#9c27b0']

    for idx, nombre in enumerate(jugadores):
        fila = df_norm[df_norm['Jugador'].str.contains(nombre, case=False, na=False)]
        if fila.empty:
            print(f"Jugador '{nombre}' no encontrado")
            continue
        fila = fila.iloc[0]
        valores = [fila[m] if m in fila.index else 0 for m in metricas]
        valores += valores[:1]
        color = colores[idx % len(colores)]
        ax.plot(angulos, valores, linewidth=2, linestyle='solid', color=color, label=nombre)
        ax.fill(angulos, valores, alpha=0.15, color=color)

    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(metricas, size=10, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.set_yticks([25, 50, 75, 100])
    ax.set_yticklabels(['P25', 'P50', 'P75', 'P100'], size=8, color='gray')
    ax.grid(color='gray', alpha=0.3)
    ax.spines['polar'].set_visible(False)

    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
    plt.title(
        f'Perfil por percentil\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
        size=13, pad=20, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_radar.png', dpi=150, bbox_inches='tight')
    plt.show()


# ── Uso — reemplazar con los jugadores y métricas relevantes del partido ──
METRICAS_RADAR = [
    'Tiros totales', 'Tiros al arco', 'Goles',
    'Entradas ganadas', 'Intercepciones', 'Faltas cometidas',
    'Centros', 'Asistencias'
]

JUGADORES_RADAR = [
    'JUGADOR 1',    # ← reemplazar
    'JUGADOR 2',    # ← reemplazar
    'JUGADOR 3',    # ← reemplazar (opcional — máximo 5)
]
# ─────────────────────────────────────────────────────────────────────────

radar_jugadores(df_activos, JUGADORES_RADAR, METRICAS_RADAR)

### 4.5 Lollipop chart — Tiros totales por jugador

**Pregunta táctica:** ¿Quiénes generaron más situaciones de peligro ofensivo?

El lollipop chart es más limpio que el barplot para datasets con muchos jugadores — reduce el ruido visual  
y enfatiza el valor exacto de cada jugador.

In [ ]:
df_ofensivo = df_activos[['Jugador', 'Posición principal', 'Minutos',
                            'Tiros totales', 'Tiros al arco',
                            'Goles', 'Asistencias']].copy()
df_ofensivo = df_ofensivo[df_activos['Tiros totales'] > 0].sort_values('Tiros totales', ascending=True)

COLORES_POS = {
    'Arquero': '#6c757d', 'Defensor': '#1a73e8',
    'Defensor central': '#1a73e8', 'Lateral izquierdo': '#4a90d9',
    'Lateral derecho': '#4a90d9', 'Mediocampista': '#34a853',
    'Mediocampista defensivo': '#2d8a47', 'Mediocampista central': '#34a853',
    'Mediocampista ofensivo': '#5cb85c', 'Delantero': '#e8341a',
    'Extremo izquierdo': '#e8341a', 'Extremo derecho': '#e8341a',
}

fig, ax = plt.subplots(figsize=(12, max(6, len(df_ofensivo) * 0.45)))

for _, row in df_ofensivo.iterrows():
    color = COLORES_POS.get(row['Posición principal'], '#adb5bd')
    ax.plot([0, row['Tiros totales']], [row['Jugador'], row['Jugador']],
            color=color, linewidth=2, alpha=0.7)
    ax.scatter(row['Tiros totales'], row['Jugador'],
               color=color, s=120, zorder=5)
    ax.text(row['Tiros totales'] + 0.05, row['Jugador'],
            f" {int(row['Tiros totales'])}", va='center', fontsize=9)

# Leyenda de posiciones
leyenda = [mpatches.Patch(color=c, label=p) for p, c in COLORES_POS.items()
           if p in df_ofensivo['Posición principal'].values]
ax.legend(handles=leyenda, loc='lower right', fontsize=8, title='Posición')

ax.set_xlabel('Tiros totales', fontsize=11)
ax.set_ylabel('')
ax.set_xlim(-0.2, df_ofensivo['Tiros totales'].max() + 1)
ax.axvline(df_ofensivo['Tiros totales'].mean(), color='gray',
           linestyle='--', alpha=0.6, label='Promedio')
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.title(f'Tiros totales por jugador\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_lollipop_tiros.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.6 Bubble chart — Tiros vs Minutos jugados

**Pregunta táctica:** ¿Quiénes fueron más incisivos en relación al tiempo que estuvieron en cancha?

- **Eje X:** minutos jugados  
- **Eje Y:** tiros totales  
- **Tamaño de burbuja:** tiros al arco  
- **Color:** posición del jugador

Un jugador con pocos minutos y muchos tiros es más incisivo que uno con 90 minutos y pocos tiros.

In [ ]:
df_bubble = df_activos[df_activos['Minutos'] >= 10].copy()

fig, ax = plt.subplots(figsize=(13, 8))

posiciones_unicas = df_bubble['Posición principal'].unique()
palette = sns.color_palette('husl', len(posiciones_unicas))
color_map = dict(zip(posiciones_unicas, palette))

for _, row in df_bubble.iterrows():
    color = color_map.get(row['Posición principal'], 'gray')
    size = max(row['Tiros al arco'] * 200, 80)
    ax.scatter(row['Minutos'], row['Tiros totales'],
               s=size, color=color, alpha=0.75, edgecolors='white', linewidth=1.5)
    ax.annotate(row['Jugador'].split()[-1],
                (row['Minutos'], row['Tiros totales']),
                textcoords='offset points', xytext=(6, 4),
                fontsize=8, color='#333333')

# Línea de promedio
ax.axhline(df_bubble['Tiros totales'].mean(), color='gray',
           linestyle='--', alpha=0.5, linewidth=1, label='Prom. tiros')
ax.axvline(45, color='#adb5bd', linestyle=':', alpha=0.7, label='Entretiempo')

# Leyenda de posiciones
leyenda = [mpatches.Patch(color=c, label=p) for p, c in color_map.items()]
ax.legend(handles=leyenda, loc='upper left', fontsize=8,
          title='Posición', framealpha=0.8)

ax.set_xlabel('Minutos jugados', fontsize=11)
ax.set_ylabel('Tiros totales', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(alpha=0.2)

# Nota sobre tamaño de burbuja
ax.text(0.98, 0.02, 'Tamaño = tiros al arco',
        transform=ax.transAxes, ha='right', fontsize=8, color='gray')

plt.title(f'Tiros vs Minutos jugados\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_bubble_tiros_minutos.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.7 Scatter chart — Perfil ofensivo vs defensivo

**Pregunta táctica:** ¿Hubo jugadores dominantes en ambas fases, o cada uno fue especialista en una?

- **Eje X:** acciones defensivas (entradas + intercepciones)  
- **Eje Y:** acciones ofensivas (tiros totales)  
- Los cuadrantes permiten clasificar a cada jugador en su rol predominante en el partido.

In [ ]:
df_scatter = df_activos[df_activos['Minutos'] >= 10].copy()
df_scatter['acciones_defensivas'] = df_scatter['Entradas ganadas'] + df_scatter['Intercepciones']

fig, ax = plt.subplots(figsize=(12, 8))

posiciones_unicas = df_scatter['Posición principal'].unique()
palette = sns.color_palette('husl', len(posiciones_unicas))
color_map = dict(zip(posiciones_unicas, palette))

for _, row in df_scatter.iterrows():
    color = color_map.get(row['Posición principal'], 'gray')
    ax.scatter(row['acciones_defensivas'], row['Tiros totales'],
               s=100, color=color, alpha=0.8, edgecolors='white', linewidth=1.5)
    ax.annotate(row['Jugador'].split()[-1],
                (row['acciones_defensivas'], row['Tiros totales']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

# Líneas de promedio como cuadrantes
med_def = df_scatter['acciones_defensivas'].mean()
med_of  = df_scatter['Tiros totales'].mean()
ax.axvline(med_def, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(med_of,  color='gray', linestyle='--', alpha=0.5, linewidth=1)

# Etiquetas de cuadrantes
xlim = ax.get_xlim()
ylim = ax.get_ylim()
ax.text(xlim[1]*0.75, ylim[1]*0.9, 'Completo', fontsize=9,
        color='#34a853', fontweight='bold', ha='center')
ax.text(xlim[1]*0.75, ylim[0]*0.5, 'Defensivo', fontsize=9,
        color='#1a73e8', fontweight='bold', ha='center')
ax.text(xlim[0]+0.1, ylim[1]*0.9, 'Ofensivo', fontsize=9,
        color='#e8341a', fontweight='bold', ha='center')
ax.text(xlim[0]+0.1, ylim[0]*0.5, 'Bajo impacto', fontsize=9,
        color='#adb5bd', fontweight='bold', ha='center')

leyenda = [mpatches.Patch(color=c, label=p) for p, c in color_map.items()]
ax.legend(handles=leyenda, loc='upper right', fontsize=8, title='Posición')

ax.set_xlabel('Acciones defensivas (entradas + intercepciones)', fontsize=11)
ax.set_ylabel('Tiros totales', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(alpha=0.2)

plt.title(f'Perfil ofensivo vs defensivo por jugador\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_scatter_of_def.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.8 Heatmap táctico — Métricas por jugador

**Pregunta táctica:** ¿Cuál fue el perfil completo de cada jugador en una sola visualización?

El heatmap táctico es la visualización más completa para análisis de partido —  
permite ver de un vistazo quién hizo qué en cada dimensión del juego.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

metricas_heatmap = [
    'Minutos', 'Tiros totales', 'Tiros al arco',
    'Goles', 'Asistencias', 'Entradas ganadas',
    'Intercepciones', 'Faltas cometidas', 'Faltas recibidas', 'Centros'
]
metricas_heatmap = [m for m in metricas_heatmap if m in df_activos.columns]

df_heat = df_activos[['Jugador', 'Posición principal'] + metricas_heatmap].copy()
df_heat = df_heat[df_heat['Minutos'] > 0].sort_values('Minutos', ascending=False)
df_heat = df_heat.set_index('Jugador')

# Normalizar por columna (0-1)
df_norm = df_heat[metricas_heatmap].copy()
for col in df_norm.columns:
    rng = df_norm[col].max() - df_norm[col].min()
    df_norm[col] = (df_norm[col] - df_norm[col].min()) / rng if rng > 0 else 0

cmap_custom = LinearSegmentedColormap.from_list(
    'futbol', ['#f8f9fa', '#74c0fc', '#1971c2', '#0b3d91']
)

fig, ax = plt.subplots(figsize=(14, max(8, len(df_norm) * 0.45)))
sns.heatmap(
    df_norm,
    annot=df_heat[metricas_heatmap].astype(int),
    fmt='d',
    cmap=cmap_custom,
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Valor normalizado (0 = mínimo, 1 = máximo del partido)'}
)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=35, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.title(f'Heatmap táctico — {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})\n'
          f'(valores reales anotados, color = intensidad relativa)',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_heatmap_tactico.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.9 Distribución de minutos por posición

**Pregunta táctica:** ¿Cómo se distribuyó el uso de recursos humanos por línea táctica?

El gráfico de torta con porcentajes muestra claramente cómo el técnico distribuyó los minutos disponibles.

In [ ]:
minutos_por_pos = df_activos.groupby('Posición principal')['Minutos'].sum().sort_values(ascending=False)

colores_pos = ['#1a73e8', '#e8341a', '#34a853', '#fbbc04', '#9c27b0',
               '#00bcd4', '#ff5722', '#607d8b']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Torta
wedges, texts, autotexts = axes[0].pie(
    minutos_por_pos.values,
    labels=minutos_por_pos.index,
    autopct='%1.1f%%',
    colors=colores_pos[:len(minutos_por_pos)],
    startangle=90,
    pctdistance=0.82,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
for text in autotexts:
    text.set_fontsize(9)
axes[0].set_title('% de minutos por posición', fontsize=12, fontweight='bold')

# Barras con minutos absolutos
bars = axes[1].barh(
    minutos_por_pos.index[::-1],
    minutos_por_pos.values[::-1],
    color=colores_pos[:len(minutos_por_pos)][::-1],
    edgecolor='white', height=0.6
)
for bar, val in zip(bars, minutos_por_pos.values[::-1]):
    axes[1].text(val + 5, bar.get_y() + bar.get_height()/2,
                 f'{int(val)} min', va='center', fontsize=9)
axes[1].set_xlabel('Minutos totales')
axes[1].set_title('Minutos totales por posición', fontsize=12, fontweight='bold')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle(f'Distribución de minutos — {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_distribucion_minutos.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.10 Lollipop chart — Rendimiento per90

**Pregunta táctica:** ¿Quiénes fueron más intensos en relación a su tiempo de juego?

Normalizar por 90 minutos permite comparar titulares y suplentes en igualdad de condiciones.

In [ ]:
df_per90 = df_activos[df_activos['Minutos'] >= 10].copy()
df_per90['tiros_per90'] = (df_per90['Tiros totales'] / df_per90['Minutos'] * 90).round(2)
df_per90['def_per90']   = (df_per90['acciones_defensivas'] / df_per90['Minutos'] * 90).round(2)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, metrica, titulo, color in zip(
    axes,
    ['tiros_per90', 'def_per90'],
    ['Tiros per90', 'Acciones defensivas per90'],
    ['#1a73e8', '#e8341a']
):
    df_sorted = df_per90.sort_values(metrica, ascending=True)
    media = df_per90[metrica].mean()

    for _, row in df_sorted.iterrows():
        c = color if row[metrica] >= media else '#adb5bd'
        ax.plot([0, row[metrica]], [row['Jugador'], row['Jugador']],
                color=c, linewidth=2, alpha=0.8)
        ax.scatter(row[metrica], row['Jugador'], color=c, s=100, zorder=5)
        ax.text(row[metrica] + 0.03, row['Jugador'],
                f" {row[metrica]:.2f}", va='center', fontsize=8)

    ax.axvline(media, color='gray', linestyle='--', alpha=0.6,
               linewidth=1, label=f'Prom: {media:.2f}')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Por 90 minutos', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.2)
    ax.legend(fontsize=9)

plt.suptitle(f'Rendimiento per90 — {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_lollipop_per90.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.11 Disciplina — Faltas cometidas vs recibidas

**Pregunta táctica:** ¿Hubo jugadores especialmente problemáticos o muy foulleados?

In [ ]:
df_disciplina = df_activos[
    (df_activos['Faltas cometidas'] > 0) |
    (df_activos['Faltas recibidas'] > 0) |
    (df_activos['Tarjetas amarillas'] > 0) |
    (df_activos['Tarjetas rojas'] > 0)
].copy().sort_values('Faltas cometidas', ascending=False)

if len(df_disciplina) == 0:
    print('Sin faltas ni tarjetas registradas en este partido')
else:
    df_disc_melt = df_disciplina[['Jugador', 'Faltas cometidas', 'Faltas recibidas']].melt(
        id_vars='Jugador', var_name='Tipo', value_name='Faltas'
    )

    plt.figure(figsize=(12, 6))
    p = sns.barplot(
        data=df_disc_melt, x='Jugador', y='Faltas', hue='Tipo',
        palette={'Faltas cometidas': '#e8341a', 'Faltas recibidas': '#1a73e8'}
    )
    for container in p.containers:
        p.bar_label(container, label_type='edge', padding=3)
    plt.title(f'Faltas cometidas vs recibidas\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
              fontweight='bold')
    plt.xlabel('')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_disciplina.png', dpi=150, bbox_inches='tight')
    plt.show()

    tarjetas = df_activos[
        (df_activos['Tarjetas amarillas'] > 0) | (df_activos['Tarjetas rojas'] > 0)
    ][['Jugador', 'Posición principal', 'Tarjetas amarillas', 'Tarjetas rojas']]

    if len(tarjetas) == 0:
        print('Sin tarjetas en este partido')
    else:
        print('--- Tarjetas ---')
        display(tarjetas)

### 4.12 Perfil individual de jugador

**⚠️ Cambiar el nombre del jugador según el análisis que querés hacer.**

In [ ]:
# ── Cambiar el nombre del jugador a analizar ─────────────────────────────
JUGADOR_ANALISIS = 'NOMBRE JUGADOR'    # ← reemplazar
# ─────────────────────────────────────────────────────────────────────────

perfil_jugador(df_activos, JUGADOR_ANALISIS)

### 4.13 Resumen ejecutivo del partido

**Síntesis automática de los hallazgos más relevantes del análisis.**

In [ ]:
df_activos['acciones_defensivas'] = df_activos['Entradas ganadas'] + df_activos['Intercepciones']
df_activos['contribucion_ofensiva'] = df_activos['Goles'] + df_activos['Asistencias']

mejor_ofensivo  = df_activos.loc[df_activos['Tiros totales'].idxmax(), 'Jugador']
mejor_defensivo = df_activos.loc[df_activos['acciones_defensivas'].idxmax(), 'Jugador']
mas_minutos     = df_activos.loc[df_activos['Minutos'].idxmax(), 'Jugador']
mas_fouls_com   = df_activos.loc[df_activos['Faltas cometidas'].idxmax(), 'Jugador']
mas_fouls_rec   = df_activos.loc[df_activos['Faltas recibidas'].idxmax(), 'Jugador']

contrib = df_activos[df_activos['contribucion_ofensiva'] > 0]
goleador = contrib.iloc[0]['Jugador'] if len(contrib) > 0 else 'Sin goles registrados'

tiros_totales   = int(df_activos['Tiros totales'].sum())
tiros_arco      = int(df_activos['Tiros al arco'].sum())
total_faltas    = int(df_activos['Faltas cometidas'].sum())
total_amarillas = int(df_activos['Tarjetas amarillas'].sum())
total_rojas     = int(df_activos['Tarjetas rojas'].sum())

print('=' * 60)
print(f'RESUMEN EJECUTIVO')
print(f'{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
print(f'{COMPETICION}')
print('=' * 60)
print(f'\n📊 ESTADÍSTICAS GENERALES')
print(f'   Tiros totales:       {tiros_totales}')
print(f'   Tiros al arco:       {tiros_arco} ({tiros_arco/tiros_totales*100:.0f}% de precisión)')
print(f'   Faltas totales:      {total_faltas}')
print(f'   Tarjetas amarillas:  {total_amarillas}')
print(f'   Tarjetas rojas:      {total_rojas}')
print(f'\n⚽ JUGADORES DESTACADOS')
print(f'   Más incisivo (tiros):    {mejor_ofensivo}')
print(f'   Más defensivo:           {mejor_defensivo}')
print(f'   Más minutos:             {mas_minutos}')
print(f'   Más faltas cometidas:    {mas_fouls_com}')
print(f'   Más faltas recibidas:    {mas_fouls_rec}')
if len(contrib) > 0:
    print(f'   Contribuciones (G+A):   {goleador}')
print('\n📁 ARCHIVOS GENERADOS')
archivos = [f for f in os.listdir(OUTPUT_PATH) if PARTIDO_ID in f]
for archivo in sorted(archivos):
    print(f'   → {OUTPUT_PATH}/{archivo}')
print('=' * 60)